# Serienindividuelle Filterung nach letzter Nachfrage

Dieses Notebook prüft, welche Store-Produkt-Reihen nach ihrer aktuellen letzten Nachfragepause historisch ungewöhnlich inaktiv wirken.

Vor der Lückenanalyse werden die Reihen auf einen Mindestnachfrageanteil von `MIN_DEMAND_SHARE_OVERALL = 10 %` gefiltert. Der Nachfrageanteil ist der Anteil beobachteter aktiver Tage mit positiver Nachfrage über die vollständige Reihe und ist bewusst vom Modellselektions-Schwellenwert bis zum ersten Origin getrennt.

Zwei disjunkte Herkunftsklassen werden über `is_fcm` und `is_pseudo` aus demselben gepoolten Datensatz verglichen:

- Pseudo (`is_fcm = false`, `is_pseudo = true`)
- FCM (`is_fcm = true`)

Datenquelle: `data/interim/transactions_dst_over_days`.


## Definition

Eine Reihe ist eine eindeutige Kombination aus `MARKT_ID` und `ARTIKEL_ID`.

Für jede Reihe wird der letzte Verkaufstag als letzter Tag mit positiver Nachfrage berechnet: `ABVERKAUFTE_MENGE_KG > 0`. Das entspricht der Demand-Spalte, die auch in der Tagesverteilung als Startkriterium verwendet wird. Als Datenstand gilt der späteste Kalendertag im jeweiligen Datensatz, nicht der letzte Verkaufstag einer einzelnen Reihe.

Die aktuelle Lücke ist:

`Datenstand - letzter positiver Verkaufstag`

Die historischen Lücken einer Reihe sind:

`aktueller positiver Verkaufstag - vorheriger positiver Verkaufstag - 1 Tag`

Der Filter kombiniert zwei Regeln. Erstens wird eine Reihe entfernt, wenn der letzte Verkauf mehr als 100 Tage zurückliegt. Zweitens wird eine Reihe entfernt, wenn ihre aktuelle Lücke deutlich größer ist als die eigene historische Referenzlücke. Dadurch bleiben dauerhaft inaktive Reihen draußen, ohne für alle aktiven Produkte dieselbe starre Lückenstruktur zu unterstellen.

Die Kontrollausgabe für Reihen ohne positive Nachfrage wird separat auf der ungefilterten Basis berechnet, damit ein Fehler in der Tagesverteilung nicht durch den Nachfrageanteilsfilter verdeckt wird.


In [ ]:
from pathlib import Path
import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent.parent
MIN_DEMAND_SHARE_OVERALL = 0.10
SOURCE_COLORS = {"Pseudo": "#d7a84b", "FCM": "#c4513b"}

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 100)

DATASETS = [
    {
        "Datensatz": "Pseudo",
        "Pfad": ROOT / "data" / "interim" / "transactions_dst_over_days",
        "Filter": "NOT is_fcm AND is_pseudo",
    },
    {
        "Datensatz": "FCM",
        "Pfad": ROOT / "data" / "interim" / "transactions_dst_over_days",
        "Filter": "is_fcm",
    },
]

SALE_CONDITION = "COALESCE(ABVERKAUFTE_MENGE_KG, 0) > 0"
SERIES_COLUMNS = ["MARKT_ID", "ARTIKEL_ID"]

con = duckdb.connect()
_ = con.execute("PRAGMA threads=8")
_ = con.execute("SET preserve_insertion_order=false")


## Basis nach Nachfrageanteilsfilter

Die erste Tabelle zeigt je Herkunftsklasse, wie viele Reihen der Mindestnachfrageanteil von 10 % entfernt. Die zweite Tabelle beschreibt die verbleibenden Reihen, auf denen die weitere Lückenanalyse basiert. Anschließend zeigen eine ECDF und ein gestapeltes Balkendiagramm die Verteilung des Nachfrageanteils und die Filterwirkung im direkten Klassenvergleich.


In [ ]:
def sql_string(value):
    return str(value).replace("'", "''")


def parquet_glob(directory):
    files = sorted(directory.glob("transactions_year_*.parquet"))
    if not files:
        raise FileNotFoundError(f"Keine Parquet-Dateien in {directory} gefunden")
    return sql_string(directory / "transactions_year_*.parquet")


def percent(part, total):
    return round(100 * part / total, 2) if total else 0.0


def load_series_sales(dataset):
    source = parquet_glob(dataset["Pfad"])
    return con.execute(
        f"""
        SELECT
            MARKT_ID,
            ARTIKEL_ID,
            MIN(CAST(DATE AS DATE)) AS erster_tag,
            MAX(CAST(DATE AS DATE)) AS letzter_tag,
            MIN(CASE WHEN {SALE_CONDITION} THEN CAST(DATE AS DATE) END) AS erster_verkauf,
            MAX(CASE WHEN {SALE_CONDITION} THEN CAST(DATE AS DATE) END) AS letzter_verkauf,
            COUNT(*)::BIGINT AS beobachtete_perioden,
            SUM(CASE WHEN {SALE_CONDITION} THEN 1 ELSE 0 END)::BIGINT AS nachfrageperioden,
            SUM(CASE WHEN {SALE_CONDITION} THEN 1 ELSE 0 END)::DOUBLE
                / COUNT(*) AS nachfrageanteil,
            SUM(CASE WHEN {SALE_CONDITION} THEN UMS_MENGE ELSE 0 END) AS menge_summe,
            SUM(CASE WHEN {SALE_CONDITION} THEN ABVERKAUFTE_MENGE_KG ELSE 0 END) AS kg_summe,
            SUM(CASE WHEN {SALE_CONDITION} THEN UMS_VK_WERT ELSE 0 END) AS umsatz_summe
        FROM read_parquet('{source}')
        WHERE MARKT_ID IS NOT NULL
          AND ARTIKEL_ID IS NOT NULL
          AND DATE IS NOT NULL
          AND {dataset['Filter']}
        GROUP BY MARKT_ID, ARTIKEL_ID
        """
    ).fetchdf()


def apply_min_demand_filter(dataset, series_sales):
    keep_mask = series_sales["nachfrageanteil"] >= MIN_DEMAND_SHARE_OVERALL
    kept = series_sales.loc[keep_mask].copy()
    removed = int((~keep_mask).sum())
    return kept, {
        "Datensatz": dataset["Datensatz"],
        "Pfad": dataset["Pfad"].relative_to(ROOT).as_posix(),
        "Mindest_Nachfrageanteil_%": 100 * MIN_DEMAND_SHARE_OVERALL,
        "Reihen_vor_Filter": len(series_sales),
        "Reihen_nach_Filter": len(kept),
        "Reihen_entfernt": removed,
        "Anteil_entfernt_%": percent(removed, len(series_sales)),
    }


def days_since_last_sale(series_sales):
    data_stand = series_sales["letzter_tag"].max()
    return data_stand, (data_stand - series_sales["letzter_verkauf"]).dt.days


def summarize_dataset(dataset, series_sales):
    data_stand, days_since_sale = days_since_last_sale(series_sales)
    series_with_sale = series_sales["letzter_verkauf"].notna().sum()
    return {
        "Datensatz": dataset["Datensatz"],
        "Pfad": dataset["Pfad"].relative_to(ROOT).as_posix(),
        "Reihen_nach_Mindestfilter": len(series_sales),
        "Reihen_mit_Verkauf": int(series_with_sale),
        "Produkte": series_sales["ARTIKEL_ID"].nunique(),
        "Filialen": series_sales["MARKT_ID"].nunique(),
        "erster_Tag": series_sales["erster_tag"].min().date(),
        "Datenstand": data_stand.date(),
        "erster_Verkauf": series_sales["erster_verkauf"].min().date(),
        "Median_Tage_seit_letztem_Verkauf": float(days_since_sale.dropna().median()),
        "Max_Tage_seit_letztem_Verkauf": int(days_since_sale.dropna().max()),
    }


series_sales_raw_by_dataset = {}
series_sales_by_dataset = {}
min_demand_filter_rows = []
overview_rows = []

for dataset in DATASETS:
    raw_series_sales = load_series_sales(dataset)
    series_sales, min_demand_row = apply_min_demand_filter(dataset, raw_series_sales)

    series_sales_raw_by_dataset[dataset["Datensatz"]] = raw_series_sales
    series_sales_by_dataset[dataset["Datensatz"]] = series_sales
    min_demand_filter_rows.append(min_demand_row)
    overview_rows.append(summarize_dataset(dataset, series_sales))

min_demand_filter_summary = pd.DataFrame(min_demand_filter_rows)
dataset_overview = pd.DataFrame(overview_rows)

display(min_demand_filter_summary)
display(dataset_overview)

zero_sale_series = pd.concat(
    [
        series_sales.assign(Datensatz=name)
        for name, series_sales in series_sales_raw_by_dataset.items()
    ],
    ignore_index=True,
)
zero_sale_series = zero_sale_series[zero_sale_series["letzter_verkauf"].isna()]
zero_sale_series = zero_sale_series[
    [
        "Datensatz",
        "MARKT_ID",
        "ARTIKEL_ID",
        "erster_tag",
        "letzter_tag",
        "beobachtete_perioden",
        "nachfrageperioden",
        "menge_summe",
        "kg_summe",
        "umsatz_summe",
    ]
].sort_values(["Datensatz", "MARKT_ID", "ARTIKEL_ID"])

print("Kontrolle vor Nachfrageanteilsfilter: Reihen ohne positive Nachfrage")
display(zero_sale_series)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for dataset_name, frame in series_sales_raw_by_dataset.items():
    values = np.sort(frame["nachfrageanteil"].dropna().to_numpy(dtype=float) * 100)
    cumulative = np.arange(1, len(values) + 1) / len(values) * 100
    axes[0].step(
        values, cumulative, where="post", linewidth=2,
        color=SOURCE_COLORS[dataset_name], label=dataset_name,
    )
axes[0].axvline(
    100 * MIN_DEMAND_SHARE_OVERALL, color="black", linestyle="--",
    linewidth=1.3, label="Mindestanteil 10 %",
)
axes[0].set_xlim(0, 50)
axes[0].set_ylim(0, 100)
axes[0].set_title("Verteilung des Nachfrageanteils")
axes[0].set_xlabel("Aktive Tage mit Nachfrage (%)")
axes[0].set_ylabel("Kumulativer Anteil der Reihen (%)")
axes[0].grid(alpha=0.25)
axes[0].legend()

filter_plot = min_demand_filter_summary.set_index("Datensatz").loc[list(SOURCE_COLORS)]
retained_pct = 100 * filter_plot["Reihen_nach_Filter"] / filter_plot["Reihen_vor_Filter"]
removed_pct = 100 - retained_pct
x = np.arange(len(filter_plot))
axes[1].bar(x, retained_pct, color=[SOURCE_COLORS[name] for name in filter_plot.index], label="Behalten")
axes[1].bar(x, removed_pct, bottom=retained_pct, color="#d9d9d9", label="Unter 10 %")
for position, name in enumerate(filter_plot.index):
    axes[1].text(position, retained_pct.loc[name] / 2, f"{int(filter_plot.loc[name, 'Reihen_nach_Filter']):,}", ha="center", va="center", color="white", fontweight="bold")
    axes[1].text(position, retained_pct.loc[name] + removed_pct.loc[name] / 2, f"{removed_pct.loc[name]:.1f}% entfernt", ha="center", va="center", fontsize=9)
axes[1].set_xticks(x, filter_plot.index)
axes[1].set_ylim(0, 100)
axes[1].set_title("Wirkung des 10%-Nachfrageanteilsfilters")
axes[1].set_ylabel("Anteil der Reihen (%)")
axes[1].grid(axis="y", alpha=0.25)
axes[1].legend(loc="lower right")

fig.tight_layout()
plt.show()


## Quantile der aktuellen Lücken und historischen Lücken je Reihe

Die aktuelle Lücke ist der Abstand zwischen Datenstand und letztem positiven Verkaufstag einer Reihe. Damit die historischen Lücken vergleichbar sind, wird auch hier zuerst je Reihe eine Kennzahl gebildet. Die Tabelle zeigt die Verteilungen dieser Serienwerte nach dem Nachfrageanteilsfilter.


In [ ]:
GAP_QUANTILES = {
    "Minimum": 0.00,
    "P10": 0.10,
    "P25": 0.25,
    "Median": 0.50,
    "P75": 0.75,
    "P90": 0.90,
    "P95": 0.95,
    "P99": 0.99,
    "Maximum": 1.00,
}


def load_historical_gap_days(dataset, eligible_series):
    source = parquet_glob(dataset["Pfad"])
    positive_days = con.execute(
        f"""
        SELECT
            MARKT_ID,
            ARTIKEL_ID,
            CAST(DATE AS DATE) AS verkaufstag
        FROM read_parquet('{source}')
        WHERE MARKT_ID IS NOT NULL
          AND ARTIKEL_ID IS NOT NULL
          AND DATE IS NOT NULL
          AND {SALE_CONDITION}
          AND {dataset['Filter']}
        GROUP BY MARKT_ID, ARTIKEL_ID, verkaufstag
        ORDER BY MARKT_ID, ARTIKEL_ID, verkaufstag
        """
    ).fetchdf()
    positive_days["verkaufstag"] = pd.to_datetime(positive_days["verkaufstag"])
    positive_days = positive_days.merge(
        eligible_series[SERIES_COLUMNS].drop_duplicates(),
        on=SERIES_COLUMNS,
        how="inner",
    )
    positive_days = positive_days.sort_values([*SERIES_COLUMNS, "verkaufstag"])
    positive_days["vorheriger_verkaufstag"] = positive_days.groupby(
        SERIES_COLUMNS,
        observed=True,
    )["verkaufstag"].shift()
    positive_days["historische_luecke_tage"] = (
        positive_days["verkaufstag"] - positive_days["vorheriger_verkaufstag"]
    ).dt.days - 1
    return positive_days.dropna(subset=["historische_luecke_tage"]).copy()


def summarize_historical_gaps_by_series(historical_gaps):
    return (
        historical_gaps.groupby(SERIES_COLUMNS, observed=True)
        .agg(
            historische_luecken=("historische_luecke_tage", "size"),
            mittlere_historische_luecke=("historische_luecke_tage", "mean"),
            median_historische_luecke=("historische_luecke_tage", "median"),
            p90_historische_luecke=(
                "historische_luecke_tage",
                lambda values: values.quantile(0.90),
            ),
            maximale_historische_luecke=("historische_luecke_tage", "max"),
        )
        .reset_index()
    )


def add_gap_quantile_row(rows, dataset_name, gap_type, values, series_count):
    values = pd.Series(values).dropna()
    row = {
        "Datensatz": dataset_name,
        "Lueckentyp": gap_type,
        "Reihen_nach_Mindestfilter": series_count,
        "Serienwerte_in_Verteilung": len(values),
    }
    for label, quantile in GAP_QUANTILES.items():
        row[label] = values.quantile(quantile) if not values.empty else pd.NA
    rows.append(row)


gap_quantile_rows = []
historical_gaps_by_dataset = {}
historical_gap_series_summary_by_dataset = {}

for dataset in DATASETS:
    series_sales = series_sales_by_dataset[dataset["Datensatz"]]
    _, current_gap_days = days_since_last_sale(series_sales)
    current_gap_days = current_gap_days.dropna()
    add_gap_quantile_row(
        gap_quantile_rows,
        dataset_name=dataset["Datensatz"],
        gap_type="Aktuelle Lücke seit letztem Verkauf",
        values=current_gap_days,
        series_count=len(series_sales),
    )

    historical_gaps = load_historical_gap_days(dataset, series_sales)
    historical_gap_series_summary = summarize_historical_gaps_by_series(historical_gaps)
    historical_gaps_by_dataset[dataset["Datensatz"]] = historical_gaps
    historical_gap_series_summary_by_dataset[dataset["Datensatz"]] = historical_gap_series_summary

    historical_summary_columns = {
        "Historische mittlere Lücke je Reihe": "mittlere_historische_luecke",
        "Historische Median-Lücke je Reihe": "median_historische_luecke",
        "Historische P90-Lücke je Reihe": "p90_historische_luecke",
        "Historische Maximallücke je Reihe": "maximale_historische_luecke",
    }
    for gap_type, column in historical_summary_columns.items():
        add_gap_quantile_row(
            gap_quantile_rows,
            dataset_name=dataset["Datensatz"],
            gap_type=gap_type,
            values=historical_gap_series_summary[column],
            series_count=len(series_sales),
        )

gap_quantiles = pd.DataFrame(gap_quantile_rows)
quantile_columns = list(GAP_QUANTILES)
display(
    gap_quantiles.style.format(
        {column: "{:.1f}" for column in quantile_columns},
        na_rep="",
    )
)


In [ ]:
def ecdf(values):
    values = np.sort(pd.Series(values).dropna().to_numpy(dtype=float))
    if len(values) == 0:
        return values, values
    y = np.arange(1, len(values) + 1) / len(values) * 100
    return values, y


fig, axes = plt.subplots(len(DATASETS), 1, figsize=(10, 4 * len(DATASETS)), sharex=False)
if len(DATASETS) == 1:
    axes = [axes]

for ax, dataset in zip(axes, DATASETS):
    dataset_name = dataset["Datensatz"]
    series_sales = series_sales_by_dataset[dataset_name]
    _, current_gap_days = days_since_last_sale(series_sales)
    historical_summary = historical_gap_series_summary_by_dataset[dataset_name]

    plot_values = {
        "Aktuelle Lücke": current_gap_days,
        "Historische Median-Lücke je Reihe": historical_summary["median_historische_luecke"],
        "Historische mittlere Lücke je Reihe": historical_summary["mittlere_historische_luecke"],
    }
    for label, values in plot_values.items():
        x, y = ecdf(values)
        ax.step(x, y, where="post", linewidth=1.8, label=label)

    combined = pd.concat([pd.Series(values).dropna() for values in plot_values.values()])
    ax.set_xlim(0, max(1, combined.quantile(0.99) * 1.05))
    ax.set_ylim(0, 100)
    ax.set_title(f"{dataset_name}: Verteilung der Lücken")
    ax.set_xlabel("Lücke ohne Verkauf (Tage)")
    ax.set_ylabel("Kumulativer Anteil (%)")
    ax.grid(alpha=0.25)
    ax.legend(loc="lower right")

fig.tight_layout()
plt.show()


## Verteilung des Lückenfaktors

Der Lückenfaktor vergleicht die aktuelle Pause einer Reihe mit ihrer eigenen mittleren historischen Pause: `aktuelle Lücke / mittlere historische Lücke`. Der Mittelwert wird aus allen historischen Lücken der jeweiligen Reihe gebildet. Die ECDF markiert den verwendeten Faktor 3; die Sensitivitätskurven zeigen zusätzlich, wie sich alternative Faktoren auf den Anteil entfernter Reihen auswirken würden.


In [ ]:
CURRENT_TO_HISTORICAL_GAP_FACTOR = 10.0
HISTORICAL_REFERENCE_COLUMN = "mittlere_historische_luecke"
HISTORICAL_REFERENCE_LABEL = "mittlere historische Lücke je Reihe"


def build_gap_factor_data(dataset):
    dataset_name = dataset["Datensatz"]
    series_sales = series_sales_by_dataset[dataset_name].copy()
    _, current_gap_days = days_since_last_sale(series_sales)
    series_sales["aktuelle_luecke_tage"] = current_gap_days
    series_sales["verkaufshistorie_tage"] = (
        series_sales["letzter_verkauf"] - series_sales["erster_verkauf"]
    ).dt.days

    historical_summary = historical_gap_series_summary_by_dataset[dataset_name]
    result = series_sales.merge(historical_summary, on=SERIES_COLUMNS, how="left")
    result["historische_referenz_luecke"] = result[HISTORICAL_REFERENCE_COLUMN]
    result["luecken_faktor"] = np.where(
        result["historische_referenz_luecke"] > 0,
        result["aktuelle_luecke_tage"] / result["historische_referenz_luecke"],
        np.where(result["aktuelle_luecke_tage"] > 0, np.inf, 0.0),
    )
    result["Datensatz"] = dataset_name
    return result


gap_factor_by_dataset = {
    dataset["Datensatz"]: build_gap_factor_data(dataset)
    for dataset in DATASETS
}

GAP_FACTOR_QUANTILES = {
    "P50": 0.50,
    "P75": 0.75,
    "P80": 0.80,
    "P85": 0.85,
    "P90": 0.90,
    "P95": 0.95,
    "P99": 0.99,
}

factor_quantile_rows = []
factor_threshold_rows = []
threshold_factors = sorted({1.5, 2.0, 2.5, 3.0, CURRENT_TO_HISTORICAL_GAP_FACTOR, 4.0, 5.0})

for dataset_name, frame in gap_factor_by_dataset.items():
    finite_factors = frame["luecken_faktor"].replace([np.inf, -np.inf], np.nan).dropna()
    quantile_row = {
        "Datensatz": dataset_name,
        "Serienwerte_in_Verteilung": len(finite_factors),
    }
    for label, quantile in GAP_FACTOR_QUANTILES.items():
        quantile_row[label] = finite_factors.quantile(quantile)
    factor_quantile_rows.append(quantile_row)

    for factor in threshold_factors:
        removed = int((frame["luecken_faktor"] > factor).sum())
        factor_threshold_rows.append({
            "Datensatz": dataset_name,
            "Faktor": factor,
            "Reihen_entfernt": removed,
            "Anteil_entfernt_%": percent(removed, len(frame)),
        })

factor_quantiles = pd.DataFrame(factor_quantile_rows)
factor_threshold_summary = pd.DataFrame(factor_threshold_rows)

display(
    factor_quantiles.style.format(
        {column: "{:.2f}" for column in GAP_FACTOR_QUANTILES},
    )
)
display(
    factor_threshold_summary.style.format({
        "Faktor": "{:.2f}",
        "Anteil_entfernt_%": "{:.2f}",
    })
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
factor_plot_limit = max(
    1.0,
    max(
        frame["luecken_faktor"].replace([np.inf, -np.inf], np.nan).quantile(0.99)
        for frame in gap_factor_by_dataset.values()
    ),
)
for dataset_name, frame in gap_factor_by_dataset.items():
    factors = np.sort(
        frame["luecken_faktor"].replace([np.inf, -np.inf], np.nan).dropna().to_numpy(dtype=float)
    )
    cumulative = np.arange(1, len(factors) + 1) / len(factors) * 100
    axes[0].step(
        factors, cumulative, where="post", linewidth=2,
        color=SOURCE_COLORS[dataset_name], label=dataset_name,
    )
axes[0].axvline(
    CURRENT_TO_HISTORICAL_GAP_FACTOR, color="black", linestyle="--",
    linewidth=1.3, label=f"Filterfaktor {CURRENT_TO_HISTORICAL_GAP_FACTOR:g}",
)
axes[0].set_xlim(0, factor_plot_limit)
axes[0].set_ylim(0, 100)
axes[0].set_title("Verteilung des Lückenfaktors (bis P99)")
axes[0].set_xlabel("Aktuelle / mittlere historische Lücke")
axes[0].set_ylabel("Kumulativer Anteil der Reihen (%)")
axes[0].grid(alpha=0.25)
axes[0].legend()

for dataset_name, frame in factor_threshold_summary.groupby("Datensatz", sort=False):
    axes[1].plot(
        frame["Faktor"], frame["Anteil_entfernt_%"], marker="o",
        linewidth=2, color=SOURCE_COLORS[dataset_name], label=dataset_name,
    )
axes[1].axvline(CURRENT_TO_HISTORICAL_GAP_FACTOR, color="black", linestyle="--", linewidth=1.3)
axes[1].set_title("Sensitivität gegenüber dem Lückenfaktor")
axes[1].set_xlabel("Angenommener Filterfaktor")
axes[1].set_ylabel("Entfernte Reihen (%)")
axes[1].grid(alpha=0.25)
axes[1].legend()

fig.tight_layout()
plt.show()


## Serienindividueller Filter

Eine Reihe wird entfernt, wenn mindestens eine der beiden Bedingungen gilt:

- Der letzte Verkauf liegt mehr als `MAX_CURRENT_GAP_DAYS` Tage zurück. Diese Regel entfernt Reihen, die am Datenende bereits sehr lange keine Nachfrage mehr hatten.
- Die aktuelle Lücke ist größer als `CURRENT_TO_HISTORICAL_GAP_FACTOR * historische Referenzlücke`. Diese Regel entfernt Reihen, deren aktuelle Pause im Verhältnis zur eigenen Historie ungewöhnlich lang ist.

Die Ergebnistabelle trennt beide Gründe, damit sichtbar bleibt, ob Reihen wegen der absoluten 100-Tage-Grenze, wegen des serienindividuellen Faktors oder wegen beider Regeln entfernt werden. Ein gestapeltes Balkendiagramm vergleicht diese Gründe nach Herkunftsklasse. Die anschließenden Streudiagramme zeigen aktuelle gegen historische Lücken zusammen mit beiden Entscheidungsgrenzen.


In [ ]:
MAX_CURRENT_GAP_DAYS = 100

individual_filter_by_dataset = {}
for dataset_name, frame in gap_factor_by_dataset.items():
    filtered_frame = frame.copy()
    filtered_frame["entfernen_wegen_aktueller_luecke"] = (
        filtered_frame["aktuelle_luecke_tage"] > MAX_CURRENT_GAP_DAYS
    )
    filtered_frame["entfernen_wegen_lueckenfaktor"] = (
        filtered_frame["aktuelle_luecke_tage"]
        > CURRENT_TO_HISTORICAL_GAP_FACTOR * filtered_frame["historische_referenz_luecke"]
    )
    filtered_frame["entfernen"] = (
        filtered_frame["entfernen_wegen_aktueller_luecke"]
        | filtered_frame["entfernen_wegen_lueckenfaktor"]
    )
    filtered_frame["Entfernungsgrund"] = np.select(
        [
            filtered_frame["entfernen_wegen_aktueller_luecke"]
            & filtered_frame["entfernen_wegen_lueckenfaktor"],
            filtered_frame["entfernen_wegen_aktueller_luecke"],
            filtered_frame["entfernen_wegen_lueckenfaktor"],
        ],
        [
            "aktuelle Lücke > 100 Tage und Lückenfaktor",
            "aktuelle Lücke > 100 Tage",
            "Lückenfaktor",
        ],
        default="behalten",
    )
    individual_filter_by_dataset[dataset_name] = filtered_frame

removed_series_by_dataset = {
    dataset_name: frame[frame["entfernen"]].copy()
    for dataset_name, frame in individual_filter_by_dataset.items()
}

individual_filter_summary = pd.DataFrame(
    [
        {
            "Datensatz": dataset_name,
            "Referenz": HISTORICAL_REFERENCE_LABEL,
            "Faktor": CURRENT_TO_HISTORICAL_GAP_FACTOR,
            "Max_aktuelle_Luecke_Tage": MAX_CURRENT_GAP_DAYS,
            "Reihen_nach_Mindestfilter": len(frame),
            "Reihen_mit_historischer_Referenz": int(frame["historische_referenz_luecke"].notna().sum()),
            "Reihen_entfernt_wegen_aktueller_Luecke": int(
                frame["entfernen_wegen_aktueller_luecke"].sum()
            ),
            "Reihen_entfernt_wegen_Lueckenfaktor": int(
                frame["entfernen_wegen_lueckenfaktor"].sum()
            ),
            "Reihen_entfernt_wegen_beider_Regeln": int(
                (
                    frame["entfernen_wegen_aktueller_luecke"]
                    & frame["entfernen_wegen_lueckenfaktor"]
                ).sum()
            ),
            "Reihen_entfernt": int(frame["entfernen"].sum()),
            "Anteil_entfernt_%": percent(int(frame["entfernen"].sum()), len(frame)),
            "Median_aktuelle_Luecke_entfernt": float(
                frame.loc[frame["entfernen"], "aktuelle_luecke_tage"].median()
            ),
            "Median_Faktor_entfernt": float(
                frame.loc[frame["entfernen"], "luecken_faktor"].replace(np.inf, np.nan).median()
            ),
        }
        for dataset_name, frame in individual_filter_by_dataset.items()
    ]
)

display(individual_filter_summary.style.format({
    "Faktor": "{:.2f}",
    "Anteil_entfernt_%": "{:.2f}",
    "Median_aktuelle_Luecke_entfernt": "{:.1f}",
    "Median_Faktor_entfernt": "{:.1f}",
}))

removal_reason_rows = []
for dataset_name, frame in individual_filter_by_dataset.items():
    absolute = frame["entfernen_wegen_aktueller_luecke"]
    factor = frame["entfernen_wegen_lueckenfaktor"]
    removal_reason_rows.append({
        "Datensatz": dataset_name,
        "Behalten": int((~frame["entfernen"]).sum()),
        "Nur >100 Tage": int((absolute & ~factor).sum()),
        "Nur Faktor": int((factor & ~absolute).sum()),
        "Beide Regeln": int((absolute & factor).sum()),
    })
removal_reason_summary = pd.DataFrame(removal_reason_rows).set_index("Datensatz").loc[list(SOURCE_COLORS)]
removal_reason_pct = removal_reason_summary.div(removal_reason_summary.sum(axis=1), axis=0) * 100

reason_colors = {
    "Behalten": "#6b9f78",
    "Nur >100 Tage": "#f2b134",
    "Nur Faktor": "#7a5195",
    "Beide Regeln": "#c4513b",
}
fig, ax = plt.subplots(figsize=(9, 4.8))
bottom = np.zeros(len(removal_reason_pct))
x = np.arange(len(removal_reason_pct))
for reason in removal_reason_pct.columns:
    values = removal_reason_pct[reason].to_numpy()
    ax.bar(x, values, bottom=bottom, color=reason_colors[reason], label=reason)
    for position, (value, base) in enumerate(zip(values, bottom)):
        if value >= 3:
            ax.text(position, base + value / 2, f"{value:.1f}%", ha="center", va="center", fontsize=9, color="white")
    bottom += values
ax.set_xticks(x, removal_reason_pct.index)
ax.set_ylim(0, 100)
ax.set_title("Ergebnis und Entfernungsgründe des serienindividuellen Filters")
ax.set_ylabel("Anteil der Reihen nach 10%-Filter (%)")
ax.grid(axis="y", alpha=0.25)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=4)
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(DATASETS), figsize=(15, 4.5), sharex=False, sharey=False)
for ax, dataset in zip(axes, DATASETS):
    dataset_name = dataset["Datensatz"]
    frame = individual_filter_by_dataset[dataset_name].dropna(
        subset=["historische_referenz_luecke", "aktuelle_luecke_tage"]
    ).copy()
    if len(frame) > 4000:
        frame = frame.sample(4000, random_state=42)
    kept = frame[~frame["entfernen"]]
    removed = frame[frame["entfernen"]]
    ax.scatter(kept["historische_referenz_luecke"], kept["aktuelle_luecke_tage"], s=8, alpha=0.22, color="#777777", label="Behalten")
    ax.scatter(removed["historische_referenz_luecke"], removed["aktuelle_luecke_tage"], s=10, alpha=0.45, color=SOURCE_COLORS[dataset_name], label="Entfernt")
    x_limit = max(1, frame["historische_referenz_luecke"].quantile(0.99))
    y_limit = max(MAX_CURRENT_GAP_DAYS * 1.1, frame["aktuelle_luecke_tage"].quantile(0.99))
    factor_x = np.linspace(0, x_limit, 200)
    ax.plot(factor_x, CURRENT_TO_HISTORICAL_GAP_FACTOR * factor_x, color="#7a5195", linestyle=":", linewidth=1.5, label="Faktorgrenze")
    ax.axhline(MAX_CURRENT_GAP_DAYS, color="#c4513b", linestyle="--", linewidth=1.3, label="100-Tage-Grenze")
    ax.set_xlim(0, x_limit)
    ax.set_ylim(0, y_limit)
    ax.set_title(dataset_name)
    ax.set_xlabel("Mittlere historische Lücke (Tage)")
    ax.grid(alpha=0.2)
axes[0].set_ylabel("Aktuelle Lücke (Tage)")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.02))
fig.suptitle("Entscheidungsgrenzen nach Herkunftsklasse", y=1.09)
fig.tight_layout()
plt.show()


## Beispiele entfernter Reihen

Gezeigt werden je drei unterschiedliche Produkte der Klassen Pseudo und FCM aus den entfernten Reihen mit mindestens 100 Tagen Verkaufshistorie. Die Auswahl deckt die absolute 100-Tage-Regel und die serienindividuelle Faktorregel ab. Jede Reihe bekommt eine eigene, bei langen Historien begrenzte x-Achse.


In [ ]:
MIN_EXAMPLE_SALES_HISTORY_DAYS = 100
N_EXAMPLES_PER_GROUP = 3
MAX_EXAMPLE_PLOT_DAYS = 730
REMOVAL_REASON_COLUMNS = [
    "entfernen_wegen_aktueller_luecke",
    "entfernen_wegen_lueckenfaktor",
]


def select_reason_balanced_examples(candidates, n):
    candidates = candidates.sort_values("aktuelle_luecke_tage").reset_index(drop=True)
    if candidates["ARTIKEL_ID"].nunique() < n:
        raise RuntimeError(
            f"Benötigt werden {n} unterschiedliche Produkte, verfügbar sind nur "
            f"{candidates['ARTIKEL_ID'].nunique()}."
        )
    selected_indices = []

    def available_pool(mask):
        selected_articles = set(
            candidates.loc[selected_indices, "ARTIKEL_ID"].tolist()
        )
        return candidates.loc[
            mask
            & ~candidates.index.isin(selected_indices)
            & ~candidates["ARTIKEL_ID"].isin(selected_articles)
        ]

    absolute_reason = candidates["entfernen_wegen_aktueller_luecke"]
    factor_reason = candidates["entfernen_wegen_lueckenfaktor"]
    reason_masks = [
        absolute_reason & ~factor_reason,
        factor_reason & ~absolute_reason,
        absolute_reason & factor_reason,
    ]
    for mask in reason_masks:
        pool = available_pool(mask)
        if pool.empty or len(selected_indices) >= n:
            continue
        target_gap = pool["aktuelle_luecke_tage"].median()
        selected_indices.append(
            (pool["aktuelle_luecke_tage"] - target_gap).abs().idxmin()
        )

    for reason_column in REMOVAL_REASON_COLUMNS:
        selected_has_reason = (
            bool(candidates.loc[selected_indices, reason_column].any())
            if selected_indices else False
        )
        if selected_has_reason or not candidates[reason_column].any():
            continue
        pool = available_pool(candidates[reason_column])
        selected_indices.append(pool["aktuelle_luecke_tage"].idxmax())

    while len(selected_indices) < n:
        remaining = available_pool(pd.Series(True, index=candidates.index)).copy()
        selected_gaps = candidates.loc[selected_indices, "aktuelle_luecke_tage"]
        remaining["gap_distance"] = remaining["aktuelle_luecke_tage"].apply(
            lambda gap: (selected_gaps - gap).abs().min()
        )
        selected_indices.append(remaining["gap_distance"].idxmax())

    return candidates.loc[selected_indices].copy()


example_data_path = DATASETS[0]["Pfad"]
selected_examples = []
for dataset in DATASETS:
    dataset_name = dataset["Datensatz"]
    candidates = removed_series_by_dataset[dataset_name].copy()
    candidates = candidates[
        candidates["verkaufshistorie_tage"] >= MIN_EXAMPLE_SALES_HISTORY_DAYS
    ].copy()
    candidates["Datensatz"] = dataset_name
    selected_examples.append(
        select_reason_balanced_examples(candidates, N_EXAMPLES_PER_GROUP)
    )
example_series = pd.concat(selected_examples, ignore_index=True)
example_series["Beispiel"] = (
    example_series.groupby("Datensatz", observed=True).cumcount() + 1
).map(lambda number: f"Beispiel {number}")

if example_series.empty:
    print("Keine entfernten Beispielreihen mit mindestens 100 Tagen Verkaufshistorie gefunden.")
else:
    con.register("example_keys", example_series[SERIES_COLUMNS])
    source = parquet_glob(example_data_path)
    example_daily = con.execute(
        f"""
        SELECT
            t.MARKT_ID,
            t.ARTIKEL_ID,
            CAST(t.DATE AS DATE) AS DATE,
            COALESCE(t.ABVERKAUFTE_MENGE_KG, 0.0)::DOUBLE AS Nachfrage_kg,
            t.ARTIKEL_BEZ
        FROM read_parquet('{source}') AS t
        INNER JOIN example_keys AS k
            ON t.MARKT_ID = k.MARKT_ID
           AND t.ARTIKEL_ID = k.ARTIKEL_ID
        WHERE t.DATE IS NOT NULL
        ORDER BY t.MARKT_ID, t.ARTIKEL_ID, t.DATE
        """
    ).fetchdf()
    con.unregister("example_keys")
    example_daily["DATE"] = pd.to_datetime(example_daily["DATE"])

    example_table = example_series[
        [
            "Beispiel",
            "Datensatz",
            "MARKT_ID",
            "ARTIKEL_ID",
            "aktuelle_luecke_tage",
            "historische_referenz_luecke",
            "luecken_faktor",
            "Entfernungsgrund",
            "entfernen_wegen_aktueller_luecke",
            "entfernen_wegen_lueckenfaktor",
            "verkaufshistorie_tage",
            "nachfrageperioden",
        ]
    ].copy()
    display(example_table.style.format({
        "aktuelle_luecke_tage": "{:.0f}",
        "historische_referenz_luecke": "{:.1f}",
        "luecken_faktor": "{:.1f}",
        "verkaufshistorie_tage": "{:.0f}",
    }))

    fig, axes = plt.subplots(
        len(example_series),
        1,
        figsize=(10, 2.9 * len(example_series)),
        sharex=False,
    )
    if len(example_series) == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, example_series.iterrows()):
        series = example_daily[
            (example_daily["MARKT_ID"] == row["MARKT_ID"])
            & (example_daily["ARTIKEL_ID"] == row["ARTIKEL_ID"])
        ].sort_values("DATE").copy()
        full_x_start = max(
            series["DATE"].min(),
            row["erster_verkauf"] - pd.Timedelta(days=7),
        )
        recent_x_start = row["letzter_tag"] - pd.Timedelta(
            days=MAX_EXAMPLE_PLOT_DAYS
        )
        tail_x_start = row["letzter_verkauf"] - pd.Timedelta(days=7)
        x_start = max(full_x_start, min(recent_x_start, tail_x_start))
        x_end = row["letzter_tag"] + pd.Timedelta(days=7)
        plot_series = series[series["DATE"] >= x_start].copy()
        positive = plot_series[plot_series["Nachfrage_kg"] > 0]
        product_name = (
            series["ARTIKEL_BEZ"].dropna().iloc[0]
            if series["ARTIKEL_BEZ"].notna().any()
            else "Unbekanntes Produkt"
        )
        title = (
            f"{row['Beispiel']} ({row['Datensatz']}): {product_name}\n"
            f"aktuelle Lücke {row['aktuelle_luecke_tage']:.0f} Tage, "
            f"mittlere historische Referenz {row['historische_referenz_luecke']:.1f} Tage, "
            f"Faktor {row['luecken_faktor']:.1f}\n"
            f"Entfernungsgrund: {row['Entfernungsgrund']}"
        )

        ax.plot(
            plot_series["DATE"],
            plot_series["Nachfrage_kg"],
            color="0.55",
            linewidth=0.9,
            label="Tagesnachfrage",
        )
        ax.scatter(
            positive["DATE"],
            positive["Nachfrage_kg"],
            color="#4c78a8",
            s=14,
            zorder=3,
        )
        ax.axvspan(
            row["letzter_verkauf"],
            row["letzter_tag"],
            color="#f58518",
            alpha=0.16,
            label="aktuelle Nachfragelücke",
        )
        ax.axvline(
            row["letzter_verkauf"],
            color="#f58518",
            linewidth=1.0,
            label="letzter Verkauf",
        )
        if row["entfernen_wegen_aktueller_luecke"]:
            absolute_limit_date = row["letzter_verkauf"] + pd.Timedelta(
                days=MAX_CURRENT_GAP_DAYS
            )
            ax.axvline(
                absolute_limit_date,
                color="#c4513b",
                linestyle="--",
                linewidth=1.4,
                label=f"absolute Grenze ({MAX_CURRENT_GAP_DAYS} Tage)",
            )
        if row["entfernen_wegen_lueckenfaktor"]:
            factor_limit_date = row["letzter_verkauf"] + pd.Timedelta(
                days=(
                    CURRENT_TO_HISTORICAL_GAP_FACTOR
                    * row["historische_referenz_luecke"]
                )
            )
            ax.axvline(
                factor_limit_date,
                color="#7a5195",
                linestyle=":",
                linewidth=1.6,
                label="serienindividuelle Faktorgrenze",
            )
        ax.set_xlim(x_start, x_end)
        ax.set_title(title, fontsize=10, loc="left")
        ax.set_ylabel("kg")
        ax.tick_params(axis="x", labelrotation=35, labelbottom=True)
        plt.setp(ax.get_xticklabels(), ha="right")
        ax.grid(alpha=0.25)
        ax.legend(loc="upper right", fontsize=8)

    fig.tight_layout(h_pad=2.0)
    plt.show()
